In [14]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# 1) Locate latest all-sites metrics JSON
results_dir = Path("../ttm_finetuning_results_mase")
all_sites_files = sorted(results_dir.glob("all_sites_metrics_*.json"), key=lambda p: p.stat().st_mtime)

if not all_sites_files:
    raise FileNotFoundError(f"No all_sites_metrics_*.json found in {results_dir.resolve()}")

latest_file = all_sites_files[-1]
print(f"Using metrics file: {latest_file}")

with latest_file.open("r") as f:
    all_sites = json.load(f)

# 2) Build one row per (site, channel)
methods = [
    ("ttm", "ttm_metrics"),
    ("mean_baseline", "mean_baseline_metrics"),
    ("median_baseline", "median_baseline_metrics"),
]
metrics = ["mse", "rmse", "mae", "mape", "r2", "smape", "mase"]

rows = []
for site_entry in sorted(all_sites, key=lambda entry: str(entry.get("site", ""))):
    site_name = site_entry.get("site")
    file_name = site_entry.get("file")

    # Channel names are consistent across all methods
    channel_names = sorted(site_entry.get("ttm_metrics", {}).get("per_channel", {}).keys())

    for channel in channel_names:
        row = {
            "site": site_name,
            "file": file_name,
            "channel": channel,
        }

        # Method-specific metric values (e.g., mse_ttm, mse_mean_baseline, ... )
        for metric in metrics:
            for method_name, method_key in methods:
                value = (
                    site_entry.get(method_key, {})
                    .get("per_channel", {})
                    .get(channel, {})
                    .get(metric, np.nan)
                )
                row[f"{metric}_{method_name}"] = value

        rows.append(row)

df_site_channel_metrics = pd.DataFrame(rows)

# 3) Order columns by metric: each with ttm / mean_baseline / median_baseline
ordered_cols = ["site", "file", "channel"]
for metric in metrics:
    ordered_cols.extend([
        f"{metric}_ttm",
        f"{metric}_mean_baseline",
        f"{metric}_median_baseline",
    ])

df_site_channel_metrics = df_site_channel_metrics[ordered_cols]
df_site_channel_metrics = df_site_channel_metrics.reset_index(drop=True)

print(f"Rows: {len(df_site_channel_metrics)}, Columns: {len(df_site_channel_metrics.columns)}")

# Optional save
output_csv = Path("site_channel_metrics_summary.csv")
df_site_channel_metrics.to_csv(output_csv, index=False)
print(f"Saved: {output_csv.resolve()}")

Using metrics file: ../ttm_finetuning_results_mase/all_sites_metrics_20260224_144848.json
Rows: 828, Columns: 24
Saved: /home/rishi/ML Projects/Air Pollution/CPCB/site_channel_metrics_summary.csv


In [15]:
df_site_channel_metrics

,site,file,channel,mse_ttm,mse_mean_baseline,mse_median_baseline,rmse_ttm,rmse_mean_baseline,rmse_median_baseline,mae_ttm,...,mape_median_baseline,r2_ttm,r2_mean_baseline,r2_median_baseline,smape_ttm,smape_mean_baseline,smape_median_baseline,mase_ttm,mase_mean_baseline,mase_median_baseline
0,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,CO (mg/m³),0.503698,0.943369,1.043430,0.709717,0.971272,1.021484,0.425529,...,248.895447,0.550892,0.158872,0.069655,88.618706,127.785149,130.735733,1.755029,2.613552,2.566001
1,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,NO2 (µg/m³),0.613744,0.902053,0.917368,0.783418,0.949764,0.957793,0.518421,...,827.282349,0.371438,0.076168,0.060483,99.983391,114.967262,116.394440,1.488792,1.938222,1.853239
2,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,Ozone (µg/m³),0.082652,0.120190,0.130285,0.287492,0.346684,0.360950,0.197778,...,248.476578,0.385927,0.107030,0.032030,60.613602,74.325119,69.781342,1.850140,2.457043,2.439250
3,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,PM10 (µg/m³),0.260042,0.349084,0.377689,0.509943,0.590834,0.614564,0.336750,...,288.301086,0.683364,0.574942,0.540112,58.044304,64.583557,64.266190,2.306567,2.794161,2.804076
4,site_113_Shadipur_Delhi_CPCB_15Min,site_113_Shadipur_Delhi_CPCB_15Min.csv,PM2.5 (µg/m³),0.308642,0.377965,0.413100,0.555555,0.614789,0.642729,0.297743,...,154.414536,0.444992,0.320332,0.257151,63.990845,71.581985,72.938881,3.101506,3.691738,3.814223
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
823,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,NO2 (µg/m³),0.211007,0.225298,0.287798,0.459355,0.474656,0.536468,0.356580,...,374.242096,0.122492,0.063061,-0.196855,94.197983,97.832062,93.827065,6.987213,7.603868,6.899241
824,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,Ozone (µg/m³),0.027731,0.034791,0.039282,0.166527,0.186523,0.198198,0.123269,...,1082.415161,0.444003,0.302460,0.212408,23.461679,26.006245,27.157185,5.888429,6.909683,7.333141
825,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,PM10 (µg/m³),0.012507,0.015528,0.015290,0.111836,0.124611,0.123654,0.067792,...,19.550724,0.669230,0.589349,0.595628,13.307977,15.924681,15.822892,3.285734,4.067734,4.062573
826,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min,site_5680_Lingaraj_Nagar_Hubballi_KSPCB_15Min.csv,PM2.5 (µg/m³),0.026930,0.032625,0.033184,0.164104,0.180625,0.182165,0.103524,...,213.839096,0.499109,0.393181,0.382786,65.816986,73.566994,73.334145,3.309927,4.035226,4.057751


In [16]:
# Metric summary per channel (mean and median across all sites)
metric_cols = [c for c in df_site_channel_metrics.columns if c not in ("site", "file", "channel")]

summary_per_channel = (
    df_site_channel_metrics
    .groupby("channel")[metric_cols]
    .agg(["mean", "median"])
)

# Flatten MultiIndex columns: (metric_method, stat) -> metric_method_stat
summary_per_channel.columns = [f"{col}_{stat}" for col, stat in summary_per_channel.columns]

# Re-order so columns are grouped by metric: metric_ttm_mean, metric_ttm_median, ...
ordered_summary_cols = []
for metric in metrics:
    for method_name, _ in methods:
        ordered_summary_cols.extend([
            f"{metric}_{method_name}_mean",
            f"{metric}_{method_name}_median",
        ])

ordered_summary_cols = [c for c in ordered_summary_cols if c in summary_per_channel.columns]
summary_per_channel = summary_per_channel[ordered_summary_cols]

summary_per_channel

,mse_ttm_mean,mse_ttm_median,mse_mean_baseline_mean,mse_mean_baseline_median,mse_median_baseline_mean,mse_median_baseline_median,rmse_ttm_mean,rmse_ttm_median,rmse_mean_baseline_mean,rmse_mean_baseline_median,...,smape_mean_baseline_mean,smape_mean_baseline_median,smape_median_baseline_mean,smape_median_baseline_median,mase_ttm_mean,mase_ttm_median,mase_mean_baseline_mean,mase_mean_baseline_median,mase_median_baseline_mean,mase_median_baseline_median
channel,,,,,,,,,,,,,,,,,,,,,
CO (mg/m³),0.607293,0.411791,0.807934,0.599096,0.853683,0.647373,0.686167,0.641709,0.808002,0.774013,...,91.942260,91.507938,88.779210,88.682064,2.080806,1.997446,2.595796,2.504935,2.563480,2.466111
NO2 (µg/m³),0.462654,0.265307,0.694469,0.432256,0.745995,0.451919,0.550527,0.515079,0.679845,0.657461,...,70.827045,73.859901,67.290814,72.011497,2.045722,1.721000,2.818275,2.333223,2.762841,2.284296
Ozone (µg/m³),0.531267,0.280845,0.912105,0.551074,0.976704,0.578192,0.604064,0.529948,0.816264,0.742344,...,98.795039,104.640862,85.420915,88.812614,1.805254,1.658423,2.738848,2.807311,2.546789,2.549121
PM10 (µg/m³),0.596535,0.433595,0.755085,0.580755,0.792419,0.618351,0.716118,0.658478,0.813115,0.762072,...,82.397509,82.569763,79.639236,82.186619,2.084134,2.054576,2.502880,2.471154,2.482800,2.458710
PM2.5 (µg/m³),0.635867,0.486194,0.781499,0.669610,0.822141,0.699973,0.735898,0.697276,0.823669,0.818296,...,77.152342,72.994461,75.070220,71.751194,2.176342,2.143669,2.574434,2.642894,2.572187,2.648623
SO2 (µg/m³),0.913621,0.446170,1.162677,0.561631,1.224072,0.586117,0.797742,0.667959,0.899247,0.749414,...,87.690180,91.106663,83.585426,86.092770,2.284311,2.044687,2.927456,2.456849,2.843854,2.392165


In [17]:
summary_per_channel.to_csv("ttm_finetuning_mase_v1.csv")